In [2]:
import os
from dotenv import load_dotenv
import json
import numpy as np
import pandas as pd
from typing import List, Dict, Any
import google.generativeai as genai # pip install google.generativeai
import pickle
from IPython.display import display, Markdown

In [3]:
# ========================
# 설정 부분 - 사용자가 수정해야 하는 부분
# ========================

# Google Gemini API 키 설정 (환경변수 또는 직접 입력)
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

# 데이터셋 파일 경로 (사용자가 변경)
DATA_FILE_PATH = "/Users/haley/Desktop/2025-1/DS1/code/need_preprocessing/GodQuestions/GodQuestions_raw_Kor_2025-04-29.xlsx"  # 엑셀 파일 경로

# 임베딩 모델 설정
EMBEDDING_MODEL = "models/embedding-001"  # 구글 임베딩 모델

# 생성 모델 설정 (이미 model로 정의되어 있지만 명시적으로)
GENERATION_MODEL = "gemini-1.5-flash"

# 벡터 데이터베이스 저장 파일
VECTOR_DB_FILE = "gemini_vector_database.pkl"

In [6]:
### Test

In [4]:
model = genai.GenerativeModel('gemini-1.5-flash')

In [5]:
for item in genai.list_models():
    print(item.name)

models/embedding-gecko-001
models/gemini-1.0-pro-vision-latest
models/gemini-pro-vision
models/gemini-1.5-pro-latest
models/gemini-1.5-pro-001
models/gemini-1.5-pro-002
models/gemini-1.5-pro
models/gemini-1.5-flash-latest
models/gemini-1.5-flash-001
models/gemini-1.5-flash-001-tuning
models/gemini-1.5-flash
models/gemini-1.5-flash-002
models/gemini-1.5-flash-8b
models/gemini-1.5-flash-8b-001
models/gemini-1.5-flash-8b-latest
models/gemini-1.5-flash-8b-exp-0827
models/gemini-1.5-flash-8b-exp-0924
models/gemini-2.5-pro-exp-03-25
models/gemini-2.5-pro-preview-03-25
models/gemini-2.5-flash-preview-04-17
models/gemini-2.5-flash-preview-05-20
models/gemini-2.5-flash-preview-04-17-thinking
models/gemini-2.5-pro-preview-05-06
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-preview-image-generation
models/gemini-2.0-flash-lite-preview

In [6]:
response = model.generate_content("성경이란 무엇인가?")
Markdown(response.text)

성경은 유대교와 기독교의 경전으로, 하나님과 인간의 관계, 인간의 역사, 구원, 윤리 등을 다루는 종교적·문학적 글들의 모음입니다.  단일 저자가 쓴 책이 아니라 수백 년에 걸쳐 여러 저자들이 다양한 문체와 장르 (시, 역사, 율법, 예언, 서신 등) 를 사용하여 기록한 66권의 책으로 구성되어 있습니다.

구성은 크게 다음과 같이 나뉩니다:

* **구약 (Old Testament):** 유대교의 경전이기도 하며, 기독교에서도 예수 그리스도의 도래를 예언하는 책으로 간주합니다. 주로 히브리어로 기록되었으며, 역사, 율법, 시, 예언서 등 다양한 장르의 책들이 포함되어 있습니다. 창세기부터 말라기까지 39권으로 구성됩니다.  주요 내용은 창조, 족장 시대, 이스라엘 민족의 역사, 율법, 예언 등입니다.

* **신약 (New Testament):** 예수 그리스도의 삶, 가르침, 죽음, 부활, 그리고 초기 기독교 공동체의 역사를 기록한 책들입니다. 주로 그리스어로 기록되었으며, 복음서, 사도행전, 서신서, 요한계시록 등으로 구성됩니다. 27권으로 구성됩니다. 주요 내용은 예수의 생애와 사역, 그의 가르침, 기독교의 확산, 교회의 설립과 성장, 그리고 미래에 대한 예언 등입니다.

성경은 단순한 역사책이나 문학작품이 아니라, 신앙과 삶의 지침서로 간주됩니다.  믿는 이들에게는 하나님의 말씀으로, 삶의 의미와 목적, 그리고 구원에 대한 진리를 담고 있다고 믿어집니다.  그러나 성경 해석은 다양한 관점과 방법이 존재하며, 각 종파나 개인의 신앙에 따라 해석이 다를 수 있다는 점도 중요합니다.


#### ============= ###

In [7]:
# API 키 설정 (중요!)
genai.configure(api_key=GEMINI_API_KEY)
print("✅ Gemini API 설정 완료!")

✅ Gemini API 설정 완료!


In [8]:
# 파일 경로 설정
excel_file = DATA_FILE_PATH
print(f"✅ 파일 경로 설정: {excel_file}")

✅ 파일 경로 설정: /Users/haley/Desktop/2025-1/DS1/code/need_preprocessing/GodQuestions/GodQuestions_raw_Kor_2025-04-29.xlsx


In [9]:
# SimpleRAG 클래스 (수정된 버전)
class SimpleRAG:
    def __init__(self):
        self.documents = []
        self.embeddings = []
        # genai가 이미 설정되어 있다고 가정하고 모델 생성
        self.model = genai.GenerativeModel('gemini-1.5-flash')
        print("🤖 SimpleRAG 초기화 완료!")
    
    def load_data(self, excel_file):
        """엑셀 파일에서 데이터 로드"""
        df = pd.read_excel(excel_file)
        print(f"📊 엑셀 컬럼: {list(df.columns)}")
        
        # 3번째 열(Question_KOR)과 4번째 열(Answer_KOR) 사용
        questions = df.iloc[:, 2].astype(str).tolist()  # 3번째 컬럼 (인덱스 2)
        answers = df.iloc[:, 3].astype(str).tolist()    # 4번째 컬럼 (인덱스 3)
        
        print(f"✅ Question 컬럼: {df.columns[2]}")
        print(f"✅ Answer 컬럼: {df.columns[3]}")
        
        # 문서 형태로 저장
        count = 0
        for q, a in zip(questions, answers):
            if q != 'nan' and a != 'nan' and len(q.strip()) > 0 and len(a.strip()) > 0:
                self.documents.append(f"질문: {q}\n답변: {a}")
                count += 1
        
        print(f"✅ {count}개 문서 로드 완료!")
        return count
    
    def create_embeddings(self):
        """문서들의 임베딩 생성"""
        print(f"🔄 {len(self.documents)}개 문서의 임베딩 생성 중...")
        
        self.embeddings = []
        for i, doc in enumerate(self.documents):
            try:
                result = genai.embed_content(
                    model="models/embedding-001",
                    content=doc
                )
                self.embeddings.append(result['embedding'])
                
                # 진행상황 표시
                if (i + 1) % 50 == 0:
                    print(f"📈 진행: {i + 1}/{len(self.documents)}")
                    
            except Exception as e:
                print(f"❌ 에러 (문서 {i}): {e}")
                self.embeddings.append([0] * 768)  # 빈 임베딩
        
        print("✅ 임베딩 생성 완료!")
    
    def search(self, query, top_k=3):
        """쿼리와 유사한 문서 찾기"""
        # 쿼리 임베딩 생성
        query_result = genai.embed_content(
            model="models/embedding-001",
            content=query
        )
        query_embedding = query_result['embedding']
        
        # 유사도 계산
        similarities = []
        for i, doc_embedding in enumerate(self.embeddings):
            # 코사인 유사도
            sim = np.dot(query_embedding, doc_embedding) / (
                np.linalg.norm(query_embedding) * np.linalg.norm(doc_embedding) + 1e-10
            )
            similarities.append((i, sim))
        
        # 유사도 순으로 정렬
        similarities.sort(key=lambda x: x[1], reverse=True)
        
        # 상위 k개 문서 반환
        top_docs = []
        for i, sim in similarities[:top_k]:
            top_docs.append({
                'document': self.documents[i],
                'similarity': sim
            })
        
        return top_docs
    
    def generate_answer(self, query, context_docs):
        """검색된 문서를 바탕으로 답변 생성"""
        # 컨텍스트 만들기
        context = "\n\n".join([doc['document'] for doc in context_docs])
        
        # 프롬프트 만들기
        prompt = f"""다음 참고자료를 바탕으로 질문에 답변해주세요:

참고자료:
{context}

질문: {query}

답변:"""
        
        # 답변 생성
        response = self.model.generate_content(prompt)
        return response.text
    
    def ask(self, question):
        """질문하기 (메인 함수)"""
        print(f"🤔 질문: {question}")
        
        # 1. 관련 문서 검색
        relevant_docs = self.search(question)
        
        # 2. 답변 생성
        answer = self.generate_answer(question, relevant_docs)
        
        # 3. 결과 출력 (마크다운으로 예쁘게)
        print(f"\n🤖 답변:")
        display(Markdown(answer))
        
        print(f"\n📚 참고 문서 (유사도):")
        for i, doc in enumerate(relevant_docs):
            print(f"{i+1}. 유사도: {doc['similarity']:.3f}")
            # 첫 100자만 미리보기
            preview = doc['document'].replace('\n', ' ')[:100]
            print(f"   내용: {preview}...")
            print()
        
        return answer

print("✅ SimpleRAG 클래스 정의 완료!")

✅ SimpleRAG 클래스 정의 완료!


In [11]:
rag = SimpleRAG()
rag.load_data(excel_file)
rag.create_embeddings()
print("🎉 RAG 시스템 준비 완료!")

🤖 SimpleRAG 초기화 완료!
📊 엑셀 컬럼: ['crawling_date', 'big_title_kor', 'Question_KOR', 'Answer_KOR', 'URL_KOR', 'Question_ENG', 'Answer_ENG', 'URL_ENG']
✅ Question 컬럼: Question_KOR
✅ Answer 컬럼: Answer_KOR
✅ 1440개 문서 로드 완료!
🔄 1440개 문서의 임베딩 생성 중...
📈 진행: 50/1440
📈 진행: 100/1440
📈 진행: 150/1440
📈 진행: 200/1440
📈 진행: 250/1440
📈 진행: 300/1440
📈 진행: 350/1440
📈 진행: 400/1440
📈 진행: 450/1440
📈 진행: 500/1440
📈 진행: 550/1440
📈 진행: 600/1440
📈 진행: 650/1440
📈 진행: 700/1440
📈 진행: 750/1440
📈 진행: 800/1440
📈 진행: 850/1440
📈 진행: 900/1440
📈 진행: 950/1440
📈 진행: 1000/1440
📈 진행: 1050/1440
📈 진행: 1100/1440
📈 진행: 1150/1440
📈 진행: 1200/1440
📈 진행: 1250/1440
📈 진행: 1300/1440
📈 진행: 1350/1440
📈 진행: 1400/1440
✅ 임베딩 생성 완료!
🎉 RAG 시스템 준비 완료!


In [12]:
# 질문하기
rag.ask("성경이란 무엇인가요?")

🤔 질문: 성경이란 무엇인가요?

🤖 답변:


제공된 자료에는 성경에 대한 설명이 없습니다.  따라서 성경이 무엇인가에 대한 질문에는 답변할 수 없습니다.



📚 참고 문서 (유사도):
1. 유사도: 0.861
   내용: 질문: 도덕 신학이란 무엇인가요? 답변: 도덕 신학이란 로마 가톨릭 교회에 의해 사용된 용어로 하나님의 임재 또는 은혜를 얻기 위해 인간이 어떻게 살아야 하는가 하는 관점에서 시작...

2. 유사도: 0.851
   내용: 질문: 하나님은 누가 창조했는가? 답변: 무신론자들과 회의론자들의 흔한 주장은, 만일 모든 만물이 어떤 것으로부터 기인되었다면, 하나님도 기인된 원인이 있어야 한다는 것입니다. 그...

3. 유사도: 0.840
   내용: 질문: 천사장들은 누구인가? 답변: "천사장"이라는 단어는 성경의 두 구절에서만 나타납니다. 데살로니가전서 4:16은 “주께서 호령과 천사장의 소리와 하나님의 나팔 소리로 친히 하...



'제공된 자료에는 성경에 대한 설명이 없습니다.  따라서 성경이 무엇인가에 대한 질문에는 답변할 수 없습니다.\n'

In [13]:
# 여러 질문 테스트
questions = [
    "하나님은 누구신가요?", 
    "예수님은 누구인가요?", 
    "기도란 무엇인가요?"
]

for q in questions:
    print("="*50)
    rag.ask(q)

🤔 질문: 하나님은 누구신가요?

🤖 답변:


제공된 자료에는 하나님이 누구신가에 대한 직접적인 답변이 없습니다.  하지만 제공된 자료를 바탕으로 하나님의 속성을 추론해 볼 수 있습니다.

제공된 자료에서 하나님은 다음과 같은 속성을 가지고 있습니다.

* **창조주:** 하나님은 우주와 그 안의 모든 것을 창조하신 분입니다.  무에서 유를 창조하신 존재로, 스스로 기인된 존재가 아닌 모든 것의 기인이십니다. (하나님은 누가 창조했는가? 답변 참조)
* **항상 존재하는 존재:**  절대적으로 아무것도 존재하지 않았을 수 없다는 논리에 따라, 하나님은 항상 존재해 온 존재입니다.
* **피조물의 범주를 넘어서는 존재:** 하나님은 피조물이나 기인된 것들의 범주에 속하지 않습니다.
* **최고 권위자:** 천사장 미가엘을 포함한 천사들을 지도하는 존재로서, 최고의 권위를 가지고 있습니다. (천사장들은 누구인가? 답변 참조)
* **개입하는 존재:**  데살로니가전서 4:16의 구절을 통해 하나님은 인간의 역사에 개입하시는 분으로 묘사됩니다.


요약하자면, 제공된 자료에 따르면 하나님은 우주 만물을 창조하시고, 항상 존재하며, 최고의 권위를 가지신 창조주이자 개입하시는 존재입니다.  그분은 피조물의 범주를 넘어서는 초월적인 존재입니다.  하지만 이는 자료에 제시된 정보만을 바탕으로 한 추론이며, 하나님의 본질에 대한 완전한 설명은 아닙니다.



📚 참고 문서 (유사도):
1. 유사도: 0.861
   내용: 질문: 도덕 신학이란 무엇인가요? 답변: 도덕 신학이란 로마 가톨릭 교회에 의해 사용된 용어로 하나님의 임재 또는 은혜를 얻기 위해 인간이 어떻게 살아야 하는가 하는 관점에서 시작...

2. 유사도: 0.851
   내용: 질문: 하나님은 누가 창조했는가? 답변: 무신론자들과 회의론자들의 흔한 주장은, 만일 모든 만물이 어떤 것으로부터 기인되었다면, 하나님도 기인된 원인이 있어야 한다는 것입니다. 그...

3. 유사도: 0.840
   내용: 질문: 천사장들은 누구인가? 답변: "천사장"이라는 단어는 성경의 두 구절에서만 나타납니다. 데살로니가전서 4:16은 “주께서 호령과 천사장의 소리와 하나님의 나팔 소리로 친히 하...

🤔 질문: 예수님은 누구인가요?

🤖 답변:


제공된 자료에는 예수님에 대한 설명이 없습니다. 따라서 질문에 답할 수 없습니다.  참고자료에 예수님에 대한 정보를 추가해 주시면 답변드리겠습니다.



📚 참고 문서 (유사도):
1. 유사도: 0.861
   내용: 질문: 도덕 신학이란 무엇인가요? 답변: 도덕 신학이란 로마 가톨릭 교회에 의해 사용된 용어로 하나님의 임재 또는 은혜를 얻기 위해 인간이 어떻게 살아야 하는가 하는 관점에서 시작...

2. 유사도: 0.851
   내용: 질문: 하나님은 누가 창조했는가? 답변: 무신론자들과 회의론자들의 흔한 주장은, 만일 모든 만물이 어떤 것으로부터 기인되었다면, 하나님도 기인된 원인이 있어야 한다는 것입니다. 그...

3. 유사도: 0.840
   내용: 질문: 천사장들은 누구인가? 답변: "천사장"이라는 단어는 성경의 두 구절에서만 나타납니다. 데살로니가전서 4:16은 “주께서 호령과 천사장의 소리와 하나님의 나팔 소리로 친히 하...

🤔 질문: 기도란 무엇인가요?

🤖 답변:


제공된 참고 자료에는 기도에 대한 답변이 없습니다.  따라서 기도에 대한 설명을 드릴 수 없습니다.



📚 참고 문서 (유사도):
1. 유사도: 0.861
   내용: 질문: 도덕 신학이란 무엇인가요? 답변: 도덕 신학이란 로마 가톨릭 교회에 의해 사용된 용어로 하나님의 임재 또는 은혜를 얻기 위해 인간이 어떻게 살아야 하는가 하는 관점에서 시작...

2. 유사도: 0.851
   내용: 질문: 하나님은 누가 창조했는가? 답변: 무신론자들과 회의론자들의 흔한 주장은, 만일 모든 만물이 어떤 것으로부터 기인되었다면, 하나님도 기인된 원인이 있어야 한다는 것입니다. 그...

3. 유사도: 0.840
   내용: 질문: 천사장들은 누구인가? 답변: "천사장"이라는 단어는 성경의 두 구절에서만 나타납니다. 데살로니가전서 4:16은 “주께서 호령과 천사장의 소리와 하나님의 나팔 소리로 친히 하...



In [ ]:
# 원하는 질문을 여기에 입력하세요
my_question = "구원이란 무엇인가요?"
rag.ask(my_question)